In [1]:
import util.data_loading
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Load in Data

In [2]:
debt_df= util.data_loading.load_raw_data('debt_2003_2025 _clean.csv')
debt_df2= util.data_loading.load_raw_data('debt_pre_2003.csv')

debt_df2= util.data_loading.load_raw_data('debt_pre_2003.csv')
debt_df2 = debt_df2.T

debt_df2.columns = debt_df2.iloc[0]
debt_df2 = debt_df2[1:]

debt_df2 = debt_df2.reset_index()
debt_df2 = debt_df2.rename(columns={"index": "quarter"})

debt_df = util.data_loading.clean_dates(debt_df)
debt_df2 = util.data_loading.clean_dates(debt_df2)

debt_all = pd.concat(
    [debt_df2, debt_df],
    axis=0,          # stack rows
    ignore_index=True
)

units_all =  util.data_loading.load_raw_e data('housing_units_all.csv')

median = util.data_loading.load_raw_data('MedianPricesofExistingDetachedHomesHistoricalData - Median Price.csv')
median_all  = util.data_loading.clean_median(median)

population = util.data_loading.load_raw_data('population_clean.csv')



Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/debt_2003_2025 _clean.csv
Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/debt_pre_2003.csv
Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/debt_pre_2003.csv
Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/housing_units_all.csv
Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/MedianPricesofExistingDetachedHomesHistoricalData - Median Price.csv
Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv


In [4]:
# median_all = median_all[median_all["Year"] >= 2000 & median_all["Year"] <= 2024]
median_all

,Mon-Yr,CA,Alameda,Amador,Butte,Calaveras,Contra-Costa,Del Norte,El Dorado,Fresno,...,Yuba,Unnamed: 55,Condo,LA Metro,Central Coast,Central Valley,Far North,Inland Empire,S.F. Bay Area,SoCal
0,1990-01-01,194952.0,226149.0,0.0,102143.0,0.0,0.0,0.0,0.0,82083.0,...,0.0,0.0,141519.0,203390.0,0.0,0.0,0.0,0.0,227366.0,0.0
1,1990-02-01,196273.0,219306.0,0.0,83333.0,0.0,0.0,0.0,0.0,87187.0,...,0.0,0.0,144965.0,211024.0,0.0,0.0,0.0,0.0,234739.0,0.0
2,1990-03-01,194856.0,225162.0,0.0,100000.0,0.0,0.0,0.0,0.0,83889.0,...,0.0,0.0,141132.0,209286.0,0.0,0.0,0.0,0.0,235337.0,0.0
3,1990-04-01,196111.0,229333.0,0.0,108000.0,0.0,0.0,0.0,0.0,85428.0,...,0.0,0.0,145707.0,210302.0,0.0,0.0,0.0,0.0,233178.0,0.0
4,1990-05-01,195281.0,232291.0,0.0,100000.0,0.0,0.0,0.0,0.0,88749.0,...,0.0,0.0,146060.0,210148.0,0.0,0.0,0.0,0.0,235881.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
424,2025-05-01,900170.0,1365000.0,440000.0,488750.0,499000.0,924950.0,505000.0,699000.0,440000.0,...,470000.0,0.0,675000.0,855000.0,1125000.0,510000.0,385000.0,610000.0,1400000.0,888000.0
425,2025-06-01,899790.0,1321000.0,412500.0,487000.0,465000.0,940000.0,405000.0,730000.0,438370.0,...,430000.0,0.0,670000.0,847970.0,1038000.0,499000.0,385000.0,605000.0,1400000.0,876840.0
426,2025-07-01,884050.0,1250000.0,415000.0,456500.0,456750.0,862500.0,369000.0,717500.0,440000.0,...,440000.0,0.0,647000.0,846500.0,1115680.0,500000.0,397000.0,589020.0,1300000.0,879000.0
427,2025-08-01,899130.0,1269000.0,469500.0,468000.0,540000.0,850000.0,352500.0,679500.0,446390.0,...,440000.0,0.0,649950.0,837040.0,1100000.0,495000.0,385000.0,600130.0,1275000.0,873480.0


In [ ]:
units_all

In [ ]:
debt_all

In [ ]:
mortgage_debt =  util.data_loading.get_debt(debt_all,"Mortgage")
mortgage_debt = mortgage_debt[(mortgage_debt["Year"] >= 2000) & (mortgage_debt["Year"] <= 2024)]
mortgage_debt

In [8]:
mortgage_debt = mortgage_debt.apply(pd.to_numeric, errors="coerce")

In [ ]:
#TODO: update so that units can be extraced based on just county name
ca_units = util.data_loading.get_unit_estimates(units_all, "California")
ca_units = ca_units[(ca_units["Year"] >= 2000) & (ca_units["Year"] <= 2024)]
ca_units

## Function for running Regression on specific cities wihtout having to run each cell

In [3]:
def regression(city, debt_type = "Mortgage"):

    # Median Price specific area
    median = util.data_loading.get_median_prices(median_all,city)
    # median = median[(median["Year"] >= 1999) & (median["Year"] <= 2024)]
    # median["median_pct_change"] = median[city].pct_change()

    # Unit Estimates specific area
    units = util.data_loading.get_unit_estimates(units_all, city + " County")
    units = units[(units["Year"] >= 1999) & (units["Year"] <= 2024)]
    units["units_pct_change"] = units["units"].pct_change()

    #Debt
    debt =  util.data_loading.get_debt(debt_all,debt_type)
    debt = debt[(debt["Year"] >= 1999) & (debt["Year"] <= 2024)]
    debt = debt.apply(pd.to_numeric, errors="coerce")
    debt[debt_type+"_pct_change"] = debt[debt_type].pct_change()


    # Population national
    population = util.data_loading.load_raw_data('population_clean.csv')
    population["pop_pct_change"] = population["population"].pct_change()

    features = (
    median.merge(debt, on="Year", how="inner")
       .merge(units, on="Year", how="inner")
    .merge(population, on="Year", how="inner")
    )

    features = features.fillna(0)
    # print("Data Types: \n", features.dtypes)
    # print(features)

    X = features[["Mortgage", "units", "population", "units_pct_change", "pop_pct_change", debt_type+"_pct_change"]]

    y = features[city]
    #
    X = sm.add_constant(X[["Mortgage", "units", "population", "units_pct_change", "pop_pct_change", debt_type+"_pct_change"]])


    model = sm.OLS(y, X).fit()
    print(model.summary())


In [12]:
regression("Los Angeles")

Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:            Los Angeles   R-squared:                       0.961
Model:                            OLS   Adj. R-squared:                  0.948
Method:                 Least Squares   F-statistic:                     74.32
Date:                Tue, 09 Dec 2025   Prob (F-statistic):           1.02e-11
Time:                        13:56:10   Log-Likelihood:                -298.69
No. Observations:                  25   AIC:                             611.4
Df Residuals:                      18   BIC:                             619.9
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------

In [45]:
regression("Orange")


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:                 Orange   R-squared:                       0.946
Model:                            OLS   Adj. R-squared:                  0.928
Method:                 Least Squares   F-statistic:                     52.38
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           2.00e-10
Time:                        17:56:47   Log-Likelihood:                -311.87
No. Observations:                  25   AIC:                             637.7
Df Residuals:                      18   BIC:                             646.3
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------

In [46]:
regression("Riverside")


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:              Riverside   R-squared:                       0.945
Model:                            OLS   Adj. R-squared:                  0.927
Method:                 Least Squares   F-statistic:                     51.81
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           2.19e-10
Time:                        17:58:06   Log-Likelihood:                -294.90
No. Observations:                  25   AIC:                             603.8
Df Residuals:                      18   BIC:                             612.3
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------

In [47]:
regression("San Bernardino")


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:         San Bernardino   R-squared:                       0.926
Model:                            OLS   Adj. R-squared:                  0.901
Method:                 Least Squares   F-statistic:                     37.38
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           3.30e-09
Time:                        17:59:33   Log-Likelihood:                -294.63
No. Observations:                  25   AIC:                             603.3
Df Residuals:                      18   BIC:                             611.8
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------

In [48]:
regression("San Diego")


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:              San Diego   R-squared:                       0.971
Model:                            OLS   Adj. R-squared:                  0.962
Method:                 Least Squares   F-statistic:                     101.4
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           7.00e-13
Time:                        18:01:10   Log-Likelihood:                -294.52
No. Observations:                  25   AIC:                             603.0
Df Residuals:                      18   BIC:                             611.6
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------

In [49]:
regression("Ventura")


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:                Ventura   R-squared:                       0.884
Model:                            OLS   Adj. R-squared:                  0.845
Method:                 Least Squares   F-statistic:                     22.82
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           1.70e-07
Time:                        18:01:49   Log-Likelihood:                -309.43
No. Observations:                  25   AIC:                             632.9
Df Residuals:                      18   BIC:                             641.4
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------

## Final Regression Model Custom indepedent variables for each city

In [13]:
def regression2(city, debt_type = "Mortgage", f=None):

    # Median Price specific area
    median = util.data_loading.get_median_prices(median_all,city)
    # median = median[(median["Year"] >= 1999) & (median["Year"] <= 2024)]
    # median["median_pct_change"] = median[city].pct_change()

    # Unit Estimates specific area
    units = util.data_loading.get_unit_estimates(units_all, city + " County")
    units = units[(units["Year"] >= 1999) & (units["Year"] <= 2024)]
    units["units_pct_change"] = units["units"].pct_change()

    #Debt
    debt =  util.data_loading.get_debt(debt_all,debt_type)
    debt = debt[(debt["Year"] >= 1999) & (debt["Year"] <= 2024)]
    debt = debt.apply(pd.to_numeric, errors="coerce")
    debt[debt_type+"_pct_change"] = debt[debt_type].pct_change()


    # Population national
    population = util.data_loading.load_raw_data('population_clean.csv')
    population["pop_pct_change"] = population["population"].pct_change()

    features = (
    median.merge(debt, on="Year", how="inner")
       .merge(units, on="Year", how="inner")
    .merge(population, on="Year", how="inner")
    )
    features = features.fillna(0)
    # print("Data Types: \n", features.dtypes)
    # print(features)

    X = features[f]

    y = features[city]
    #
    X = sm.add_constant(X[f])


    model = sm.OLS(y, X).fit()
    print(model.summary())


In [14]:
regression2("Los Angeles",f = [ "units", "units_pct_change",  "Mortgage_pct_change"])

Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:            Los Angeles   R-squared:                       0.957
Model:                            OLS   Adj. R-squared:                  0.951
Method:                 Least Squares   F-statistic:                     157.0
Date:                Tue, 09 Dec 2025   Prob (F-statistic):           1.53e-14
Time:                        13:56:22   Log-Likelihood:                -299.88
No. Observations:                  25   AIC:                             607.8
Df Residuals:                      21   BIC:                             612.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------

In [54]:
regression2("Orange",f = [ "units", "population", "Mortgage_pct_change"])

Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:                 Orange   R-squared:                       0.937
Model:                            OLS   Adj. R-squared:                  0.928
Method:                 Least Squares   F-statistic:                     103.7
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           9.43e-13
Time:                        19:25:56   Log-Likelihood:                -313.80
No. Observations:                  25   AIC:                             635.6
Df Residuals:                      21   BIC:                             640.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------

In [55]:
regression2("Riverside",f = ["Mortgage", "units", "population", "units_pct_change", "pop_pct_change"])

Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:              Riverside   R-squared:                       0.945
Model:                            OLS   Adj. R-squared:                  0.931
Method:                 Least Squares   F-statistic:                     65.63
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           2.53e-11
Time:                        19:27:15   Log-Likelihood:                -294.90
No. Observations:                  25   AIC:                             601.8
Df Residuals:                      19   BIC:                             609.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
---------------------------

In [56]:
regression2("San Bernardino",f = ["Mortgage", "units", "population", "units_pct_change", "pop_pct_change"])


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:         San Bernardino   R-squared:                       0.925
Model:                            OLS   Adj. R-squared:                  0.906
Method:                 Least Squares   F-statistic:                     47.03
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           4.76e-10
Time:                        19:29:25   Log-Likelihood:                -294.70
No. Observations:                  25   AIC:                             601.4
Df Residuals:                      19   BIC:                             608.7
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
---------------------------

In [57]:
regression2("San Diego", f = ["Mortgage", "units", "population", "Mortgage_pct_change"])


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:              San Diego   R-squared:                       0.964
Model:                            OLS   Adj. R-squared:                  0.957
Method:                 Least Squares   F-statistic:                     134.6
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           3.71e-14
Time:                        19:32:05   Log-Likelihood:                -297.27
No. Observations:                  25   AIC:                             604.5
Df Residuals:                      20   BIC:                             610.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------

In [60]:
regression2("Ventura", f = [ "Mortgage","population","units_pct_change", "Mortgage_pct_change"])
# ["Mortgage", "units", "population", "units_pct_change", "pop_pct_change", debt_type+"_pct_change"]

Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:                Ventura   R-squared:                       0.877
Model:                            OLS   Adj. R-squared:                  0.853
Method:                 Least Squares   F-statistic:                     35.76
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           7.54e-09
Time:                        19:38:16   Log-Likelihood:                -310.11
No. Observations:                  25   AIC:                             630.2
Df Residuals:                      20   BIC:                             636.3
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------

In [62]:
regression2("Imperial", f = ["Mortgage", "units", "population", "pop_pct_change", "Mortgage_pct_change"])


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
                            OLS Regression Results                            
Dep. Variable:               Imperial   R-squared:                       0.900
Model:                            OLS   Adj. R-squared:                  0.873
Method:                 Least Squares   F-statistic:                     34.08
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           7.48e-09
Time:                        19:40:07   Log-Likelihood:                -302.85
No. Observations:                  25   AIC:                             617.7
Df Residuals:                      19   BIC:                             625.0
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------